In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import date

# Initialize SparkSession
spark = SparkSession.builder.appName("JobTitleGrouping").getOrCreate()

# Create a sample DataFrame 'df' for demonstration
data = [
    (1, 'Cashier', date(2026, 1, 1), date(2026, 2, 1)),
    (1, 'Cashier', date(2026, 2, 1), date(2026, 3, 1)),
    (1, 'Receptionist', date(2026, 4, 1), date(2026, 5, 1)),
    (1, 'Receptionist', date(2026, 5, 1), date(2026, 6, 1)),
    (1, 'Receptionist', date(2026, 6, 1), date(2026, 7, 1)),
    (1, 'Cashier', date(2026, 8, 1), date(2026, 9, 1)),
    (1, 'Cashier', date(2026, 9, 1), date(2026, 10, 1))
]

columns = ["PersonID", "JobTitle", "FromDate", "ToDate"]
df = spark.createDataFrame(data, columns)

# 1. Define the window to look at the previous row
# We sort by date to ensure the correct sequence
window_spec = Window.partitionBy("PersonID").orderBy("FromDate")

# 2. Create a flag that marks 1 when the role changes, and 0 if it is the same as the previous one
df_flagged = df.withColumn(
    "change_flag",
    F.when(
        F.col("JobTitle") != F.lag("JobTitle").over(window_spec), 1
    ).otherwise(0)
)

df_flagged.show()
# 3. Create a group ID by calculating the cumulative sum of the flags.
# This ensures that consecutive rows with the same role have the same 'group_id'
df_grouped = df_flagged.withColumn(
    "group_id",
    F.sum("change_flag").over(window_spec)
)

df_grouped.show()
# 4. Group by the generated ID and take the Min(FromDate) and Max(ToDate)
result = (df_grouped.groupBy("PersonID", "JobTitle", "group_id")
    .agg(
        F.min("FromDate").alias("FromDate"),
        F.max("ToDate").alias("ToDate")
    ) ).drop("group_id")

result.show()

# # Stop SparkSession (optional, but good practice)
# spark.stop()

+--------+------------+----------+----------+-----------+
|PersonID|    JobTitle|  FromDate|    ToDate|change_flag|
+--------+------------+----------+----------+-----------+
|       1|     Cashier|2026-01-01|2026-02-01|          0|
|       1|     Cashier|2026-02-01|2026-03-01|          0|
|       1|Receptionist|2026-04-01|2026-05-01|          1|
|       1|Receptionist|2026-05-01|2026-06-01|          0|
|       1|Receptionist|2026-06-01|2026-07-01|          0|
|       1|     Cashier|2026-08-01|2026-09-01|          1|
|       1|     Cashier|2026-09-01|2026-10-01|          0|
+--------+------------+----------+----------+-----------+

+--------+------------+----------+----------+-----------+--------+
|PersonID|    JobTitle|  FromDate|    ToDate|change_flag|group_id|
+--------+------------+----------+----------+-----------+--------+
|       1|     Cashier|2026-01-01|2026-02-01|          0|       0|
|       1|     Cashier|2026-02-01|2026-03-01|          0|       0|
|       1|Receptionist|202

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Initialize SparkSession
spark = SparkSession.builder.appName("JobTitleGrouping").getOrCreate()

# Sample data
data = [
    (1, 'Cashier', '2026-01-01', '2026-02-01'),
    (1, 'Cashier', '2026-02-01', '2026-03-01'),
    (1, 'Receptionist', '2026-04-01', '2026-05-01'),
    (1, 'Receptionist', '2026-05-01', '2026-06-01'),
    (1, 'Receptionist', '2026-06-01', '2026-07-01'),
    (1, 'Cashier', '2026-08-01', '2026-09-01'),
    (1, 'Cashier', '2026-09-01', '2026-10-01')
]

# Create DataFrame
columns = ["PersonID", "JobTitle", "FromDate", "ToDate"]
df = spark.createDataFrame(data, columns)

# Convert string dates to date type
df = df.withColumn("FromDate", F.col("FromDate").cast("date")) \
       .withColumn("ToDate", F.col("ToDate").cast("date"))

# Define a window to calculate when the job title changes
window_spec = Window.partitionBy("PersonID").orderBy("FromDate")

# Create a flag to identify changes in JobTitle
df = df.withColumn(
    "change_flag",
    F.when(F.col("JobTitle") != F.lag("JobTitle").over(window_spec), 1).otherwise(0)
)

# Assign a group ID by calculating the cumulative sum of the change_flag
df = df.withColumn(
    "group_id",
    F.sum("change_flag").over(window_spec.rowsBetween(Window.unboundedPreceding, 0))
)

# Group by PersonID, JobTitle, and group_id to merge date ranges
result = df.groupBy("PersonID", "JobTitle", "group_id") \
    .agg(
        F.min("FromDate").alias("FromDate"),
        F.max("ToDate").alias("ToDate")
    ) \
    .drop("group_id") \
    .orderBy("PersonID", "FromDate")

# Show the result
result.show()

# Stop SparkSession (optional, but good practice)
spark.stop()


+--------+------------+----------+----------+
|PersonID|    JobTitle|  FromDate|    ToDate|
+--------+------------+----------+----------+
|       1|     Cashier|2026-01-01|2026-03-01|
|       1|Receptionist|2026-04-01|2026-07-01|
|       1|     Cashier|2026-08-01|2026-10-01|
+--------+------------+----------+----------+

